In [33]:
# imports 
import os 


In [34]:
# load grounding files 
## SLAKE
root = '/staging/users/tpadhi1/Multimodal-Uncertainty-Quantification/runs_slake/llava_med_slake'
grounding_with_llama32 = 'grounding_with_llama32'
grounding_with_gdsam = 'grounding_with_gdsam'

## GDSAM
vqa_ = '/staging/users/tpadhi1/Multimodal-Uncertainty-Quantification/runs_vqa6/llava_vqa_yes_gsam_grounding_random_1000_temp_05'
grounding_with_vqa_llama32 = os.path.join(vqa_, 'grounding_with_llama32')
grounding_with_gdsam = os.path.join(vqa_, 'grounding')
grounding_with_gdsam_llama32 = os.path.join(vqa_, 'grounding_with_gdsam_llama32')
os.makedirs(grounding_with_gdsam_llama32, exist_ok=True)
# grounding_with_gdsam = 'grounding_with_gdsam'

# grounding with both llama3.2 and gdsam
# grounding_vqa = os.path.join(root, vqa_gdsam, grounding_with_llama32)
# groundin_slake = os.path.join(root, vqa_gdsam, grounding_with_gdsam)

# load grounding files and extract questions
questions_for_which_we_have_grounding_llama32 = []
questions_for_which_we_have_grounding_gdsam = [] 

files_vqa_gr_llama32 = os.listdir(grounding_with_vqa_llama32)
files_vqa_gr_gdsam = os.listdir(grounding_with_gdsam)

question_ids_vqa_gr_llama32 = [file.split('_')[-1].split('.')[0] for file in files_vqa_gr_llama32]
question_ids_vqa_gr_gdsam = [file.split('_')[-1].split('.')[0] for file in files_vqa_gr_gdsam]
    

In [35]:
# question_ids_vqa_gr_llama32
# question_ids_vqa_gr_gdsam
# now i have to check common questions in both the lists
common_questions = set(question_ids_vqa_gr_llama32).intersection(set(question_ids_vqa_gr_gdsam))
print(len(common_questions))

496


In [36]:
import pickle

for file in files_vqa_gr_llama32:
    # load the file, its a pickle file
    # with open(os.path.join(grounding_with_gdsam, file), 'rb') as f:
    #     grounding = pickle.load(f) 
    
    # now load the corresponding file from llama3.2
    with open(os.path.join(grounding_with_vqa_llama32, file), 'rb') as f:
        grounding_llama32 = pickle.load(f)
        
    # load the corresponding file from gdsam
    path = os.path.join(grounding_with_gdsam, file)
    with open(path, 'rb') as f:
        grounding_gdsam = pickle.load(f)
        
    # check if the grounding gdsam has the key 'grounding_with_gd_score'
    # print(grounding_gdsam.keys())
    # calculate llama32 grounding score 
    # llama32_score = grounding_llama32.get('score', 0)
    llama_32_scores_all_responses = [] # here we will store the llama32 score response wise to ca
    grounding_gdsam_scores_all_responses = [] # here we will store the grounding gdsam score response wise for averaging later
    for key in grounding_llama32.keys():
        if 'response' in key:
            response = grounding_llama32.get(key, {})
            response_gdsam = grounding_gdsam.get(key, {})
            # print(response.keys())
            # print(response_gdsam.keys())
            # if 'decoded_output' in response:
            #     decoded_output = response['decoded_output']
            #     if len(decoded_output.split()) < 3:
            #         continue
            if 'llama_32_response' in response:
                llama_text = response.get('llama_32_response', '').lower().strip()
                llama_text = llama_text.replace('.<|eot_id|>', '').replace('<|eot_id|>', '').strip()
                if llama_text.startswith("yes"):
                    llama_score = 1.0
                else:
                    llama_score = 0.0
                llama_32_scores_all_responses.append(llama_score)
            # also get the grounding sam score
            if 'grounding_with_gd_score' in response_gdsam:
                grounding_gdsam_scores_all_responses.append(response_gdsam.get('grounding_with_gd_score', None))

    if llama_32_scores_all_responses:
        grounding_llama32_score_avg = sum(llama_32_scores_all_responses) / len(llama_32_scores_all_responses)
    else:
        grounding_llama32_score_avg = None
        
    print(grounding_gdsam_scores_all_responses)
    # check if the grounding gdsam score list has any NOne values, if yes, then remove them
    if None in grounding_gdsam_scores_all_responses:
        grounding_gdsam_scores_all_responses = [score for score in grounding_gdsam_scores_all_responses if score is not None]
    
    # calculate the average of the grounding gdsam scores
    grounding_gdsam_scores_avg = sum(grounding_gdsam_scores_all_responses) / len(grounding_gdsam_scores_all_responses)
    # print llaama32 score for grounding, and log in grounding gdsam 
    grounding_gdsam['grounding_gdsam_score'] = grounding_gdsam_scores_avg
    grounding_gdsam['grounding_llama32_score'] = grounding_llama32_score_avg
    # print(grounding_gdsam)
    print(grounding_gdsam.keys())
    # break
    # save the grounding gdsam file to the new directory
    with open(os.path.join(grounding_with_gdsam_llama32, file), 'wb') as f:
        pickle.dump(grounding_gdsam, f)
       

[0.3828333914279938, 0.3954259753227234, 0.3961849510669708, 0.5191861391067505, 0.3849699795246124, 0.37731122970581055, 0.3849699795246124, 0.37433314323425293, 0.3849699795246124, 0.38873404264450073, 0.3954259753227234, 0.5191861391067505, 0.4153730273246765, 0.47446373105049133, 0.523085355758667, 0.4834807813167572, 0.4863467514514923, 0.37307605147361755, 0.37433314323425293, 0.3954259753227234]
dict_keys(['question_id', 'response_0', 'response_1', 'response_2', 'response_3', 'response_4', 'response_5', 'response_6', 'response_7', 'response_8', 'response_9', 'response_10', 'response_11', 'response_12', 'response_13', 'response_14', 'response_15', 'response_16', 'response_17', 'response_18', 'response_19', 'grounding_gdsam_score', 'grounding_llama32_score'])
[0.6705003976821899, 0.47359636425971985, 0.5607337951660156, 0.44257625937461853, 0.8534248471260071, 0.5196889042854309, 0.8065480589866638, 0.7066967487335205, 0.7018152475357056, 0.8625832796096802, 0.4809357225894928, 0.